In [32]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


data = pd.read_csv('data/application_train_FE_baked.csv')

In [24]:
# New Features

def add_engineered_features(df):
    eps = 1e-6
    df = df.copy()
    df["credit_to_income"]   = df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_income"]  = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_credit"]  = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)
    df["income_per_member"]  = df["AMT_INCOME_TOTAL"] / np.maximum(df["CNT_FAM_MEMBERS"], 1)
    df["income_per_child"]   = df["AMT_INCOME_TOTAL"] / (1 + df["CNT_CHILDREN"])
    df["dsr_monthly"]        = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"]/12.0 + eps)

    df["age_years"]          = -df["DAYS_BIRTH"] / 365.25
    df["reg_years"]          = -df["DAYS_REGISTRATION"] / 365.25
    df["id_years"]           = -df["DAYS_ID_PUBLISH"] / 365.25
    df["phone_years"]        = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df["contactability"]     = df["FLAG_PHONE"] + df["FLAG_EMAIL"]

    df["prev_interest_per_credit"] = df["prev_interest_mean"] / (df["prev_amt_credit_mean"] + eps)
    df["prev_payments_ratio"]      = df["prev_last3_cnt_payment_mean"] / (df["prev_cnt_payment_mean"] + eps)
    df["recent_approval_momentum"] = df["prev_last3_approved_rate"] - df["prev_last5_approved_rate"]
    df["recent_credit_growth"]     = df["prev_last3_amt_credit_mean"] - df["prev_last5_amt_credit_mean"]
    df["prev_rate_spread"]         = df["prev_rate_mean_all"] - df["prev_rate_med_all"]

    df["region_pop_log"]     = np.log1p(df["REGION_POPULATION_RELATIVE"])
    df["region_rating_x_pop"] = df["REGION_RATING"] * df["REGION_POPULATION_RELATIVE"]

    df["log_income"]         = np.log1p(df["AMT_INCOME_TOTAL"])
    df["log_credit"]         = np.log1p(df["AMT_CREDIT"])
    df["log_annuity"]        = np.log1p(df["AMT_ANNUITY"])
    df["ext2_sq"]            = df["EXT_SOURCE_2"] ** 2
    return df

data_fe = add_engineered_features(data)

c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [25]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data_fe, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data_fe, test_size=0.2, random_state=42, stratify=data_fe['TARGET'])

# # For custom stratification, create bins for continuous variables first
# data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
# data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# # Create a combined stratification column
# data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# # Now stratify by the combined column
# train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

In [27]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    if P == 0 or N == 0:
        return 0.5
    order = np.argsort(-y_score, kind='mergesort')
    y_sorted = y_true[order]
    s_sorted = y_score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    changes = np.where(np.diff(s_sorted) != 0)[0]
    cut_idx = np.r_[changes, len(s_sorted) - 1]
    tpr = tp[cut_idx] / P
    fpr = fp[cut_idx] / N
    tpr = np.r_[0.0, tpr, 1.0]
    fpr = np.r_[0.0, fpr, 1.0]

    return float(np.trapz(tpr, fpr))

In [28]:
# Cross-validation
def cross_validate(model, train_data, val_data, cv=5):
    X = train_data.drop(columns=['TARGET']).values
    y = train_data['TARGET'].values
    val_X = val_data.drop(columns=['TARGET']).values
    val_y = val_data['TARGET'].values


    fold_size = len(X) // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else len(X)
        
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        if hasattr(model, "predict_proba"):
            y_pred_prob = model.predict_proba(X_val_fold)[:, 1]
        else:
            y_pred_prob = model.decision_function(X_val_fold)
            y_pred_prob = (y_pred_prob - y_pred_prob.min()) / (y_pred_prob.max() - y_pred_prob.min())
        

        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, y_pred_prob))

    # Evaluate on validation set
    model.fit(X, y)
    y_val_pred = model.predict(val_X)
    if hasattr(model, "predict_proba"):
        y_val_pred_prob = model.predict_proba(val_X)[:, 1]
    else:
        y_val_pred_prob = model.decision_function(val_X)
        y_val_pred_prob = (y_val_pred_prob - y_val_pred_prob.min()) / (y_val_pred_prob.max() - y_val_pred_prob.min())

    val_metrics = {
        'accuracy': accuracy(val_y, y_val_pred),
        'recall': recall(val_y, y_val_pred),
        'precision': precision(val_y, y_val_pred),
        'specificity': specificity(val_y, y_val_pred),
        'f1_score': f1_score(val_y, y_val_pred),
        'roc_auc': roc_auc(val_y, y_val_pred_prob)
    }
    
    # Calculate average coefficients accross folds if applicable
    coefs = None
    if hasattr(model, 'coef_'):
        coefs = model.coef_
    avg_metrics = {key: np.mean(value) for key, value in metrics.items()}
    return avg_metrics, coefs, val_metrics

In [45]:
# Models
# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced'
)

# SVC
svm_model = LinearSVC(
    class_weight='balanced',
    C=1.0,
    tol=1e-3,
    max_iter=5000,
    dual='auto'
)

# LDA
lda_model = LDA(
    solver='lsqr',
    shrinkage='auto',
    priors = [0.5, 0.5]
)


In [ ]:
# Run Models
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

feature_sets = {
    'Model_1': predictors1,
    'Model_2': predictors2,
    'Model_3': predictors3,
    'Model_4': predictors4,
    'Model_5': predictors5,
    'Model_6': predictors6,
    'Model_7': predictors7,
    'Model_8': predictors8
}

# Model 1
train_df = train_data_random[feature_sets['Model_1'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_1'] + [target]].copy()
log_reg_metrics1 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics1

# Model 2
train_df = train_data_random[feature_sets['Model_2'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_2'] + [target]].copy()
log_reg_metrics2 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics2

# Model 3
train_df = train_data_random[feature_sets['Model_3'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_3'] + [target]].copy()
log_reg_metrics3 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics3

# Model 4
train_df = train_data_random[feature_sets['Model_4'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_4'] + [target]].copy()
log_reg_metrics4 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics4

# Model 5
train_df = train_data_random[feature_sets['Model_5'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_5'] + [target]].copy()
log_reg_metrics5 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics5

# Model 6
train_df = train_data_random[feature_sets['Model_6'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_6'] + [target]].copy()
log_reg_metrics6 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics6

# Model 7
train_df = train_data_random[feature_sets['Model_7'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_7'] + [target]].copy()
log_reg_metrics7 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics7

# Model 8
train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
log_reg_metrics8 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics8



({'accuracy': 0.6593975681050773,
  'recall': 0.5881506974625612,
  'precision': 0.13419805091978257,
  'specificity': 0.665664423948644,
  'f1_score': 0.21852620906673997,
  'roc_auc': 0.6755293681178995},
 array([[-0.52060962, -0.03801985, -0.00719262,  0.31714708, -0.19476804]]),
 {'accuracy': 0.6607352366053969,
  'recall': 0.5978771177791385,
  'precision': 0.13448735019973368,
  'specificity': 0.6661885282190229,
  'f1_score': 0.21958167778694054,
  'roc_auc': 0.6771002391176619})

In [ ]:
print(log_reg_metrics1)
print(log_reg_metrics2)
print(log_reg_metrics3)
print(log_reg_metrics4)
print(log_reg_metrics5)
print(log_reg_metrics6)
print(log_reg_metrics7)
print(log_reg_metrics8)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

({'accuracy': 0.5972672704731187, 'recall': 0.589576784040746, 'precision': 0.11439291350183178, 'specificity': 0.5979490329152117, 'f1_score': 0.19160274670150318, 'roc_auc': 0.6264029430670857}, array([[ 0.01068565, -0.15279518,  0.10908297,  0.30108053,  0.07355062,
        -0.11183769, -0.02765984,  0.20267392,  0.26620971,  0.08077512]]), {'accuracy': 0.5920186416373354, 'recall': 0.58420085731782, 'precision': 0.11066429510478695, 'specificity': 0.5926968779330252, 'f1_score': 0.18607977634017103, 'roc_auc': 0.6195962747197654})
({'accuracy': 0.6434565956216003, 'recall': 0.5214274042859364, 'precision': 0.11726184361016304, 'specificity': 0.6541997124123213, 'f1_score': 0.19146103873198886, 'roc_auc': 0.6168168596238812}, array([[-0.09531739, -0.05259296,  0.23225922, -0.24091227,  0.05721039,
        -0.11658918,  0.06612066,  0.18675333,  0.07854221]]), {'accuracy': 0.6410181201929345, 'recall': 0.511328842620943, 'precision': 0.11313852129533444, 'specificity': 0.652269386743

In [46]:
x7_plus = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "prev_approved_rate","prev_app_count","DAYS_BIRTH","DAYS_REGISTRATION","REGION_RATING",
    # + engineered
    "credit_to_income","annuity_to_income","annuity_to_credit",
    "income_per_member","dsr_monthly",
    "age_years","reg_years",
    "prev_interest_per_credit","prev_payments_ratio","recent_approval_momentum",
    "prev_rate_spread","region_rating_x_pop","ext2_sq"
]

x8_plus = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_INCOME_TOTAL","debt_ratio","prev_approved_rate",
    # + engineered
    "credit_to_income","annuity_to_income","annuity_to_credit",
    "income_per_member","dsr_monthly",
    "age_years","reg_years",
    "prev_interest_per_credit","recent_approval_momentum","ext2_sq"
]

train_df = train_data_random[x7_plus + [target]].copy()
val_df   = val_data_random[x7_plus + [target]].copy()
log_reg_metrics9 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics9)

train_df = train_data_random[x8_plus + [target]].copy()
val_df   = val_data_random[x8_plus + [target]].copy()
log_reg_metrics10 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics10)


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.6541260187020845, 'recall': 0.613152422704111, 'precision': 0.13630286942884232, 'specificity': 0.6577325488172815, 'f1_score': 0.22302198464286754, 'roc_auc': 0.6887530866917946}, array([[-5.04508170e-01, -1.00642685e-01,  1.16070621e-01,
        -6.90374826e-03,  2.91596873e-01, -2.18066349e-01,
        -3.79305750e-02,  2.23912388e-01,  3.36405441e-02,
         6.89586653e-02, -3.16142363e-04, -3.58301682e-02,
        -1.35478438e-04,  6.71543937e-03,  2.91897711e-03,
        -6.13038708e-04, -9.21027902e-05, -7.13046265e-06,
         5.57713046e-05,  3.27017385e-02, -4.74690689e-03,
         1.66905007e-02, -1.92431410e-02]]), {'accuracy': 0.6521476991265807, 'recall': 0.6180853235354153, 'precision': 0.13455385709207252, 'specificity': 0.6551027997662434, 'f1_score': 0.22099770098164434, 'roc_auc': 0.6851525322304075})


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

({'accuracy': 0.6496081025660126, 'recall': 0.5979663926260403, 'precision': 0.13218548676073189, 'specificity': 0.6541461348743196, 'f1_score': 0.2165060142444765, 'roc_auc': 0.6755985414383648}, array([[-5.55434212e-01, -3.55771585e-02, -1.82379250e-02,
         3.16109192e-01, -1.94406892e-01, -3.35940568e-04,
        -3.65590570e-02, -1.64354109e-04,  2.15538837e-02,
         2.97973258e-03, -3.16219015e-02, -1.47285035e-02,
        -6.69903463e-06,  4.87462181e-02, -3.41166169e-02]]), {'accuracy': 0.6505507756485465, 'recall': 0.6068585425597061, 'precision': 0.13218033078427885, 'specificity': 0.6543413200162921, 'f1_score': 0.21707860246066224, 'roc_auc': 0.6775282712239359})


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [50]:
# get all columns except ID
train_all = train_data_random.drop(columns=['SK_ID_CURR'])
val_all = val_data_random.drop(columns=['SK_ID_CURR'])
log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values